# SDXL LoRA Training — CS-UAP Evaluation
## Run A (clean) and Run B (cloaked) portrait training

### Before running, in Kaggle notebook settings:
- **Enable GPU** (Settings → Accelerator → GPU T4 x2 or P100)
- **Enable internet access** (Settings → Internet → On) — required for
  `pip install`, `git clone`, and downloading the SDXL base model

### Expected Kaggle Dataset input structure
Upload your portrait datasets as a Kaggle Dataset, then attach it to this
notebook. Expected layout once attached (appears under `/kaggle/input/`):

```
/kaggle/input/<your-dataset-slug>/
  clean_person/
    10_ohwx person/
      img_001.jpg
      img_001.txt
      ...
  cloaked_person/
    10_ohwx person/
      img_001.jpg
      img_001.txt
      ...
```

The SDXL base model is **not** uploaded manually — it's downloaded directly
from Hugging Face inside this notebook (Step 3), since a ~7GB manual upload
to a Kaggle Dataset is slower and less reproducible than pulling it fresh
each run.

All outputs are written to `/kaggle/working/`, which Kaggle persists and
lets you download when the session ends.

## Step 0 — Verify GPU and environment

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                       "--format=csv"], capture_output=True, text=True).stdout
      or "No GPU detected — enable GPU in Settings before continuing.")

import os
print("\nContents of /kaggle/input/:")
if os.path.exists("/kaggle/input"):
    for item in os.listdir("/kaggle/input"):
        print(" ", item)
else:
    print("  (none found — attach your dataset via 'Add Input' first)")

## Step 1 — Install pinned dependencies

Installed once, no reinstall later in the notebook. The original notebook
uninstalled and reinstalled these same packages a second time in a later
cell (for the now-removed diffusers alternative pipeline) — that redundancy
is gone here.

**After this cell finishes, restart the session** (Kaggle: Run → Restart
Session), then continue from Step 2. This matches the original notebook's
requirement and is still necessary — some of these packages don't take
effect in the same running kernel that installed them.

In [ ]:
import subprocess, sys
python = sys.executable

print("Installing pinned dependencies for kohya_ss compatibility...")

subprocess.run([python, "-m", "pip", "uninstall", "-y",
    "diffusers", "transformers", "huggingface-hub", "accelerate", "peft"])

subprocess.run([python, "-m", "pip", "install", "-q",
    "diffusers==0.25.1",
    "transformers==4.38.2",
    "huggingface-hub==0.21.4",
    "accelerate==0.27.2",
    "peft==0.9.0",
    "safetensors>=0.4.0",
    "bitsandbytes>=0.41.0",
    "xformers",
    "einops",
    "omegaconf",
    "toml",
    "ftfy",
    "albumentations",
    "timm",
    "voluptuous",
])

print("\nDone. NOW RESTART THE SESSION (Run -> Restart Session).")
print("After restarting, skip this cell and continue from Step 2.")

## Step 2 — Clone kohya_ss (sd-scripts)

In [ ]:
import os, subprocess

KOHYA_DIR = "/kaggle/working/kohya_ss"

if os.path.exists(KOHYA_DIR):
    subprocess.run(["rm", "-rf", KOHYA_DIR])
    print("Removed old kohya_ss clone")

print("Cloning kohya_ss...")
result = subprocess.run([
    "git", "clone", "-q", "--depth", "1", "-b", "v23.1.6",
    "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR
])

if result.returncode != 0:
    print("Tagged version failed, cloning main...")
    subprocess.run(["git", "clone", "-q", "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR])

trainer = os.path.join(KOHYA_DIR, "sdxl_train_network.py")
if os.path.exists(trainer):
    print(f"Trainer found: {trainer}")
else:
    print(f"Trainer NOT found at {trainer} -- check the clone above")

## Step 2b — Install kohya_ss requirements and verify imports

In [ ]:
import subprocess, sys, os

python = sys.executable
KOHYA_DIR = "/kaggle/working/kohya_ss"
req_file = os.path.join(KOHYA_DIR, "requirements.txt")

if os.path.exists(req_file):
    print("Installing kohya_ss requirements...")
    subprocess.run([python, "-m", "pip", "install", "-q", "-r", req_file])
else:
    print("requirements.txt not found -- skipping (already installed in Step 1)")

print("\nVerifying imports...")
try:
    import diffusers, transformers, accelerate
    print(f"  diffusers   : {diffusers.__version__}")
    print(f"  transformers: {transformers.__version__}")
    print(f"  accelerate  : {accelerate.__version__}")
    print("All imports OK")
except ImportError as e:
    print(f"Import error: {e}")
    print("Restart the session and re-run from Step 1.")

## Step 3 — Download SDXL base model

Pulled directly from Hugging Face rather than uploaded as a Kaggle Dataset
-- reproducible without a manual multi-GB upload step, and cached so
re-running this cell doesn't re-download.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

MODEL_LOCAL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

if not os.path.exists(MODEL_LOCAL_PATH):
    print("Downloading SDXL base model (this can take a while)...")
    downloaded_path = hf_hub_download(
        repo_id="stabilityai/stable-diffusion-xl-base-1.0",
        filename="sd_xl_base_1.0.safetensors",
    )
    shutil.copy(downloaded_path, MODEL_LOCAL_PATH)
    print(f"Model saved to {MODEL_LOCAL_PATH}")
else:
    print(f"Model already present at {MODEL_LOCAL_PATH}")

## Step 4 — Configure run and verify dataset

Set `RUN = "a"` for clean images or `RUN = "b"` for cloaked images, then
run this cell. **This is the single place that controls which dataset gets
trained** -- the training cell in Step 6 reads these variables directly,
rather than hardcoding its own copy of them (which is what the original
notebook's Run A / Run B cells did, making the `RUN` toggle here have no
actual effect on them).

Change `RUN` and re-run this cell, then re-run Step 6, to switch between
Run A and Run B -- no need to duplicate the training cell.

In [ ]:
import os

RUN = "a"   # "a" for clean, "b" for cloaked -- change this and re-run to switch runs

INPUT_DATASET_SLUG = "csuap-portraits"  # <- set this to your attached Kaggle dataset's folder name
INPUT_BASE = f"/kaggle/input/{INPUT_DATASET_SLUG}"

DATASET_PATH = f"{INPUT_BASE}/{'clean_person' if RUN == 'a' else 'cloaked_person'}"
MODEL_PATH   = "/kaggle/working/sd_xl_base_1.0.safetensors"
OUTPUT_DIR   = f"/kaggle/working/outputs/lora_run_{RUN}"
OUTPUT_NAME  = f"lora_run_{RUN}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"RUN        : {RUN.upper()}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Output dir : {OUTPUT_DIR}\n")

dataset_ok = os.path.exists(DATASET_PATH)
print(f"Dataset exists: {'yes' if dataset_ok else 'NO'} -- {DATASET_PATH}")

if not dataset_ok:
    print("\nDataset folder not found. Check that:")
    print("  1. You attached your Kaggle Dataset via 'Add Input'")
    print("  2. INPUT_DATASET_SLUG above matches its actual folder name")
    print(f"\nListing {INPUT_BASE} ...")
    if os.path.exists(INPUT_BASE):
        for item in os.listdir(INPUT_BASE):
            print(" ", item)
    else:
        print(f"  {INPUT_BASE} does not exist -- dataset not attached")
else:
    subfolders = [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))]
    print(f"Subfolders: {subfolders}")
    for sub in subfolders:
        sub_path = os.path.join(DATASET_PATH, sub)
        files = os.listdir(sub_path)
        jpgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        txts = [f for f in files if f.lower().endswith(".txt")]
        status = "ready" if jpgs and txts else ("no images" if not jpgs else "no captions")
        print(f"  [{sub}] : {len(jpgs)} images, {len(txts)} captions -- {status}")

## Step 5 (optional) — Limit dataset to N images

The original notebook referenced folders like `kohya_limited_cloak` in
later cells, but no cell actually created them -- training would only have
worked if that folder was created manually outside the notebook, which
wasn't documented or reproducible. This cell actually implements it.

Skip this cell if you want to train on the full dataset as-is; the training
cell in Step 6 will use `DATASET_PATH` from Step 4 by default, or the
limited subset created here if you run it.

In [ ]:
import os, shutil

LIMIT_N_IMAGES = 30  # set to None to skip limiting

def make_limited_dataset(source_dir: str, dest_dir: str, n: int) -> str:
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)

    subfolders = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    if not subfolders:
        raise FileNotFoundError(f"No instance subfolder found in {source_dir}")
    instance_folder = subfolders[0]  # e.g. "10_ohwx person"

    src_instance = os.path.join(source_dir, instance_folder)
    dst_instance = os.path.join(dest_dir, instance_folder)
    os.makedirs(dst_instance, exist_ok=True)

    images = sorted(f for f in os.listdir(src_instance) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    selected = images[:n]

    for img_name in selected:
        stem = os.path.splitext(img_name)[0]
        txt_name = f"{stem}.txt"
        shutil.copy(os.path.join(src_instance, img_name), os.path.join(dst_instance, img_name))
        src_txt = os.path.join(src_instance, txt_name)
        if os.path.exists(src_txt):
            shutil.copy(src_txt, os.path.join(dst_instance, txt_name))
        else:
            print(f"  warning: no caption file found for {img_name}")

    print(f"Copied {len(selected)} images (of {len(images)} available) to {dst_instance}")
    return dest_dir


if LIMIT_N_IMAGES is not None:
    limited_dir = f"/kaggle/working/limited_dataset_run_{RUN}"
    DATASET_PATH = make_limited_dataset(DATASET_PATH, limited_dir, LIMIT_N_IMAGES)
    print(f"\nDATASET_PATH updated to: {DATASET_PATH}")
else:
    print("Skipped -- training on full dataset at:", DATASET_PATH)

## Step 6 — Run LoRA training

Single parameterized cell -- reads `RUN`, `DATASET_PATH`, `MODEL_PATH`,
`OUTPUT_DIR`, `OUTPUT_NAME` from Step 4 (and Step 5, if you ran it). To
train Run B after Run A: go back to Step 4, set `RUN = "b"`, re-run Step 4
(and Step 5 if used), then re-run this cell -- no separate duplicated cell
needed, unlike the original notebook's Run A/Run B cells which hardcoded
their own copies of these variables independently of the `RUN` toggle.

**Parameters** (matches Table 5 of the thesis): Network Dim 16, Alpha 8,
Steps 500, LR 1e-4, Resolution 512x512, Optimizer AdamW8bit, seed 42.

In [ ]:
import subprocess, sys, os

python = sys.executable
KOHYA_DIR = "/kaggle/working/kohya_ss"

cmd = [
    python, "-m", "accelerate.commands.launch",
    "--mixed_precision", "fp16",
    "--num_processes", "1",
    os.path.join(KOHYA_DIR, "sdxl_train_network.py"),

    # Model
    "--pretrained_model_name_or_path", MODEL_PATH,

    # Dataset
    "--train_data_dir", DATASET_PATH,
    "--resolution", "512,512",
    "--enable_bucket",
    "--bucket_reso_steps", "64",
    "--caption_extension", ".txt",

    # Output
    "--output_dir", OUTPUT_DIR,
    "--output_name", OUTPUT_NAME,
    "--save_model_as", "safetensors",
    "--save_every_n_steps", "100",

    # Training
    "--max_train_steps", "500",
    "--train_batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--save_precision", "fp16",
    "--seed", "42",

    # Learning rates
    "--learning_rate", "1e-4",
    "--unet_lr", "1e-4",
    "--text_encoder_lr", "0",
    "--lr_scheduler", "constant_with_warmup",
    "--lr_warmup_steps", "50",

    # Optimizer
    "--optimizer_type", "AdamW8bit",

    # LoRA
    "--network_module", "networks.lora",
    "--network_dim", "16",
    "--network_alpha", "8",
    "--network_train_unet_only",

    # Memory
    "--gradient_checkpointing",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--cache_text_encoder_outputs",
    "--cache_text_encoder_outputs_to_disk",
    "--lowram",
    "--no_half_vae",
    "--max_data_loader_n_workers", "1",
]

print(f"Starting LoRA Run {RUN.upper()}")
print(f"Model  : {MODEL_PATH}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output : {OUTPUT_DIR}\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=KOHYA_DIR)
for line in process.stdout:
    print(line, end="")
process.wait()

print(f"\nExit code: {process.returncode}")
if process.returncode == 0:
    saved = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".safetensors")]
    print(f"Training complete. Saved: {saved}")
else:
    print("Training failed -- check output above")

## Step 7 — Verify output files

In [ ]:
import os

print(f"Output directory: {OUTPUT_DIR}\n")
files = sorted(os.listdir(OUTPUT_DIR)) if os.path.exists(OUTPUT_DIR) else []

if files:
    for f in files:
        path = os.path.join(OUTPUT_DIR, f)
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f:50s}  {size:.1f} MB")
    safetensors = [f for f in files if f.endswith(".safetensors")]
    if safetensors:
        print(f"\n{len(safetensors)} LoRA model(s) saved successfully")
        print(f"Final model: {safetensors[-1]}")
    else:
        print("\nNo .safetensors files found -- training may have failed")
else:
    print("Output directory is empty or does not exist")

print("\nTo run the other LoRA (A <-> B):")
print('  1. Go to Step 4, set RUN = "b" (or "a")')
print("  2. Re-run Step 4 (and Step 5 if you use dataset limiting)")
print("  3. Re-run Step 6")